In [55]:
import os
import gc
import random
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.linear_model import Ridge

from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


FILE_PATH = "/Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/pca500_lag1_lag2.parquet"

OUTPUT_PATH = "/Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/tabnet_val_prediction.csv"

parquet_file = pq.ParquetFile(FILE_PATH)
all_cols = parquet_file.schema_arrow.names

print("전체 row 수:", parquet_file.metadata.num_rows)
print("전체 컬럼 수:", len(all_cols))
print("앞 컬럼 10개:", all_cols[:10])
print("뒤 컬럼 10개:", all_cols[-10:])

전체 row 수: 587583
전체 컬럼 수: 1503
앞 컬럼 10개: ['pca_0', 'pca_1', 'pca_2', 'pca_3', 'pca_4', 'pca_5', 'pca_6', 'pca_7', 'pca_8', 'pca_9']
뒤 컬럼 10개: ['pca_493_lag2', 'pca_494_lag2', 'pca_495_lag2', 'pca_496_lag2', 'pca_497_lag2', 'pca_498_lag2', 'pca_499_lag2', 'era', 'target', 'prediction']


In [46]:
# Feature / Target 설정


base_cols = [
    c for c in all_cols
    if c.startswith("pca_") and not c.endswith("_lag1") and not c.endswith("_lag2")
]

lag1_cols = [
    c for c in all_cols
    if c.startswith("pca_") and c.endswith("_lag1")
]

lag2_cols = [
    c for c in all_cols
    if c.startswith("pca_") and c.endswith("_lag2")
]

FEATURES = base_cols + lag1_cols + lag2_cols

print("base PCA 개수:", len(base_cols))
print("lag1 개수:", len(lag1_cols))
print("lag2 개수:", len(lag2_cols))
print("총 feature 개수:", len(FEATURES))
if len(FEATURES) != 1500:
    print("주의: feature 개수가 1500개가 아닙니다. 컬럼명을 다시 확인하세요.")

if "target_cyrusd_20" in all_cols:
    TARGET_COL = "target_cyrusd_20"
elif "target" in all_cols:
    TARGET_COL = "target"
else:
    raise ValueError("target_cyrusd_20 또는 target 컬럼이 없습니다.")

ERA_COL = "era"


base PCA 개수: 500
lag1 개수: 500
lag2 개수: 500
총 feature 개수: 1500


In [47]:
use_cols = FEATURES + [ERA_COL, TARGET_COL]

df = pd.read_parquet(FILE_PATH, columns=use_cols)

df[FEATURES] = (
    df[FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype("float32")
)

df[TARGET_COL] = df[TARGET_COL].astype("float32")

df = df[df[TARGET_COL].notna()].copy()

print("df shape:", df.shape)
display(df.head())

gc.collect()


df shape: (587583, 1502)


,pca_0,pca_1,pca_2,pca_3,pca_4,pca_5,pca_6,pca_7,pca_8,pca_9,...,pca_492_lag2,pca_493_lag2,pca_494_lag2,pca_495_lag2,pca_496_lag2,pca_497_lag2,pca_498_lag2,pca_499_lag2,era,target
0,2.157082,6.083780,-10.034975,-3.417653,-5.101716,0.658862,9.538446,-4.302553,7.393528,-2.881266,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,460,0.75
1,-6.917748,8.086533,-1.229872,1.661412,-2.063528,0.038389,-0.797849,2.407780,5.794246,9.493385,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,460,0.75
2,-9.775057,-4.105815,-7.142505,-5.733612,0.522592,-1.688702,0.250018,9.843674,7.669866,9.348386,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,460,0.50
3,-19.487883,-13.354597,-9.048046,0.205841,3.183959,8.305212,-7.106100,7.675611,8.001382,4.287366,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,460,0.75
4,7.916985,-4.583646,-7.021014,-10.736472,9.984262,-3.021521,-1.213559,-1.623956,1.568986,-6.605861,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,460,0.25


1062

In [48]:
# Era 기준 Split

unique_eras = sorted(df[ERA_COL].unique())
split_point = int(len(unique_eras) * 0.8)

train_eras = unique_eras[:split_point]   # 앞 80%
valid_eras = unique_eras[split_point:]   # 뒤 20%

train_df = df[df[ERA_COL].isin(train_eras)].reset_index(drop=True)
valid_df = df[df[ERA_COL].isin(valid_eras)].reset_index(drop=True)

print("전체 era 개수:", len(unique_eras))
print("train era 개수:", len(train_eras))
print("valid era 개수:", len(valid_eras))
print("train era 범위:", min(train_eras), "~", max(train_eras))
print("valid era 범위:", min(valid_eras), "~", max(valid_eras))
print("train rows:", len(train_df))
print("valid rows:", len(valid_df))


X_train = train_df[FEATURES].values.astype("float32")
y_train = train_df[TARGET_COL].values.astype("float32").reshape(-1, 1)

X_valid = valid_df[FEATURES].values.astype("float32")
y_valid = valid_df[TARGET_COL].values.astype("float32").reshape(-1, 1)

val_era = valid_df[ERA_COL].values
val_target = valid_df[TARGET_COL].values.astype("float32")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

전체 era 개수: 115
train era 개수: 92
valid era 개수: 23
train era 범위: 460 ~ 551
valid era 범위: 552 ~ 574
train rows: 461940
valid rows: 125643
X_train: (461940, 1500)
y_train: (461940, 1)
X_valid: (125643, 1500)
y_valid: (125643, 1)


In [ ]:
# Scaling


CHUNK_SIZE = 50000

scaler = StandardScaler()

for start in range(0, X_train.shape[0], CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, X_train.shape[0])
    scaler.partial_fit(np.array(X_train[start:end], dtype=np.float32))

    print(f"Scaler fit 진행: {end} / {X_train.shape[0]}")

Scaler fit 진행: 50000 / 461940
Scaler fit 진행: 100000 / 461940
Scaler fit 진행: 150000 / 461940
Scaler fit 진행: 200000 / 461940
Scaler fit 진행: 250000 / 461940
Scaler fit 진행: 300000 / 461940
Scaler fit 진행: 350000 / 461940
Scaler fit 진행: 400000 / 461940
Scaler fit 진행: 450000 / 461940
Scaler fit 진행: 461940 / 461940
Scaler fit 완료


In [ ]:
# Scaling된 memmap 생성

MEMMAP_DIR = "./model_memmap"
os.makedirs(MEMMAP_DIR, exist_ok=True)

X_train_scaled_path = os.path.join(MEMMAP_DIR, "X_train_scaled.dat")
X_valid_scaled_path = os.path.join(MEMMAP_DIR, "X_valid_scaled.dat")

X_train_scaled = np.memmap(
    X_train_scaled_path,
    dtype="float32",
    mode="w+",
    shape=X_train.shape
)

X_valid_scaled = np.memmap(
    X_valid_scaled_path,
    dtype="float32",
    mode="w+",
    shape=X_valid.shape
)

for start in range(0, X_train.shape[0], CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, X_train.shape[0])

    X_chunk = np.array(X_train[start:end], dtype=np.float32)
    X_chunk = scaler.transform(X_chunk).astype(np.float32)

    X_train_scaled[start:end] = X_chunk

    print(f"X_train scaling 진행: {end} / {X_train.shape[0]}")

for start in range(0, X_valid.shape[0], CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, X_valid.shape[0])

    X_chunk = np.array(X_valid[start:end], dtype=np.float32)
    X_chunk = scaler.transform(X_chunk).astype(np.float32)

    X_valid_scaled[start:end] = X_chunk

    print(f"X_valid scaling 진행: {end} / {X_valid.shape[0]}")

X_train_scaled.flush()
X_valid_scaled.flush()

X_train_scaled = np.memmap(
    X_train_scaled_path,
    dtype="float32",
    mode="r+",
    shape=X_train.shape
)

X_valid_scaled = np.memmap(
    X_valid_scaled_path,
    dtype="float32",
    mode="r+",
    shape=X_valid.shape
)


X_train scaling 진행: 50000 / 461940
X_train scaling 진행: 100000 / 461940
X_train scaling 진행: 150000 / 461940
X_train scaling 진행: 200000 / 461940
X_train scaling 진행: 250000 / 461940
X_train scaling 진행: 300000 / 461940
X_train scaling 진행: 350000 / 461940
X_train scaling 진행: 400000 / 461940
X_train scaling 진행: 450000 / 461940
X_train scaling 진행: 461940 / 461940
X_valid scaling 진행: 50000 / 125643
X_valid scaling 진행: 100000 / 125643
X_valid scaling 진행: 125643 / 125643


In [51]:
# 평가 함수
val_target = np.array(y_valid).reshape(-1).astype(np.float32)
val_era_arr = np.array(val_era).reshape(-1)

def numerai_corr_arrays(era_arr, target_arr, pred_arr):
    df_eval = pd.DataFrame({
        "era": era_arr,
        "target": target_arr,
        "prediction": pred_arr
    })

    def era_corr(sub):
        return np.corrcoef(
            sub["prediction"].rank(pct=True),
            sub["target"]
        )[0, 1]

    return df_eval.groupby("era").apply(era_corr)

def evaluate_and_save(model_name, val_prediction):
    per_era_corr = numerai_corr_arrays(
        val_era_arr,
        val_target,
        val_prediction
    )

    mean_corr = per_era_corr.mean()
    std_corr = per_era_corr.std()
    sharpe = mean_corr / std_corr if std_corr != 0 else 0

    rmse = np.sqrt(mean_squared_error(val_target, val_prediction))
    mae = mean_absolute_error(val_target, val_prediction)

    print(f"\n========== {model_name} Validation Result ==========")
    print(f"RMSE: {rmse:.6f}")
    print(f"MAE: {mae:.6f}")
    print(f"per-era CORR mean: {mean_corr:.6f}")
    print(f"per-era CORR std:  {std_corr:.6f}")
    print(f"Sharpe:            {sharpe:.6f}")
    print("===================================================")

    output_path = f"{model_name}_val_prediction.csv"

    val_output = pd.DataFrame({
        "era": val_era_arr,
        "prediction": val_prediction
    })

    val_output.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("저장 완료:", output_path)

    return {
        "model": model_name,
        "rmse": rmse,
        "mae": mae,
        "mean_corr": mean_corr,
        "std_corr": std_corr,
        "sharpe": sharpe,
        "output_path": output_path
    }

results = []

In [52]:
# Ridge


ridge_model = SGDRegressor(
    loss="squared_error",
    penalty="l2",
    alpha=1e-4,
    learning_rate="invscaling",
    eta0=0.01,
    power_t=0.25,
    max_iter=1,
    tol=None,
    random_state=SEED,
    average=True
)

EPOCHS_RIDGE = 10
BATCH_SIZE = 50000

y_train_arr = np.array(y_train).reshape(-1).astype(np.float32)


for epoch in range(EPOCHS_RIDGE):
    indices = np.arange(X_train_scaled.shape[0])
    np.random.shuffle(indices)

    epoch_loss = []

    for start in range(0, len(indices), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(indices))
        idx = indices[start:end]

        X_batch = np.array(X_train_scaled[idx], dtype=np.float32)
        y_batch = y_train_arr[idx]

        ridge_model.partial_fit(X_batch, y_batch)

    print(f"Ridge epoch {epoch + 1}/{EPOCHS_RIDGE} 완료")

ridge_pred = ridge_model.predict(X_valid_scaled).astype(np.float32)

results.append(
    evaluate_and_save("ridge", ridge_pred)
)

gc.collect()

Ridge epoch 1/10 완료
Ridge epoch 2/10 완료
Ridge epoch 3/10 완료
Ridge epoch 4/10 완료
Ridge epoch 5/10 완료
Ridge epoch 6/10 완료
Ridge epoch 7/10 완료
Ridge epoch 8/10 완료
Ridge epoch 9/10 완료
Ridge epoch 10/10 완료

========== ridge Validation Result ==========
RMSE: 496234904.937174
MAE: 394116416.000000
per-era CORR mean: 0.006368
per-era CORR std:  0.012970
Sharpe:            0.490978
저장 완료: ridge_val_prediction.csv


0

In [67]:
# 위 릿지가 너무 낮아 알파 직접 설정 


def ridge_eval(pred):
    per_era_corr = numerai_corr_arrays(
        val_era_arr,
        val_target,
        pred
    )

    mean_corr = per_era_corr.mean()
    std_corr = per_era_corr.std()
    sharpe = mean_corr / std_corr if std_corr != 0 else 0

    rmse = np.sqrt(mean_squared_error(val_target, pred))
    mae = mean_absolute_error(val_target, pred)

    return mean_corr, std_corr, sharpe, rmse, mae

alpha_list = [
    1,
    10,
    100,
    300,
    1000,
    3000,
    10000,
    30000,
    100000,
    300000,
    1000000,
    3000000,
    10000000,
    30_00000,
    100000000,
    300000000,
    1000000000
]

ridge_results = []
best_ridge_model = None
best_ridge_pred = None
best_score = -999

for alpha in alpha_list:
    print(f"\n===== Ridge alpha={alpha} 학습 시작 =====")

    ridge_model = Ridge(
        alpha=alpha,
        fit_intercept=True,
        solver="lsqr",
        max_iter=5000,
        tol=1e-4,
        random_state=42
    )

    ridge_model.fit(X_train_scaled, y_train_arr)

    pred = ridge_model.predict(X_valid_scaled).reshape(-1).astype(np.float32)

    mean_corr, std_corr, sharpe, rmse, mae = ridge_eval(pred)
    mean_corr_flip, std_corr_flip, sharpe_flip, rmse_flip, mae_flip = ridge_eval(-pred)

    if sharpe_flip > sharpe:
        final_pred = -pred
        final_mean_corr = mean_corr_flip
        final_std_corr = std_corr_flip
        final_sharpe = sharpe_flip
        final_rmse = rmse_flip
        final_mae = mae_flip
        flipped = True
    else:
        final_pred = pred
        final_mean_corr = mean_corr
        final_std_corr = std_corr
        final_sharpe = sharpe
        final_rmse = rmse
        final_mae = mae
        flipped = False

    print(f"alpha={alpha}")
    print(f"mean corr: {final_mean_corr:.6f}")
    print(f"std corr:  {final_std_corr:.6f}")
    print(f"sharpe:    {final_sharpe:.6f}")
    print(f"RMSE:      {final_rmse:.6f}")
    print(f"MAE:       {final_mae:.6f}")

    ridge_results.append({
        "model": "Ridge",
        "alpha": alpha,
        "mean corr": final_mean_corr,
        "std corr": final_std_corr,
        "sharpe": final_sharpe,
        "rmse": final_rmse,
        "mae": final_mae,
    })


ridge_results_df = pd.DataFrame(ridge_results).sort_values(
    "sharpe",
    ascending=False
).reset_index(drop=True)

display(ridge_results_df)


===== Ridge alpha=1 학습 시작 =====
alpha=1
mean corr: 0.040059
std corr:  0.013306
sharpe:    3.010650
RMSE:      0.223106
MAE:       0.156396

===== Ridge alpha=10 학습 시작 =====
alpha=10
mean corr: 0.040059
std corr:  0.013306
sharpe:    3.010654
RMSE:      0.223106
MAE:       0.156396

===== Ridge alpha=100 학습 시작 =====
alpha=100
mean corr: 0.040060
std corr:  0.013307
sharpe:    3.010525
RMSE:      0.223105
MAE:       0.156395

===== Ridge alpha=300 학습 시작 =====
alpha=300
mean corr: 0.040059
std corr:  0.013305
sharpe:    3.010737
RMSE:      0.223105
MAE:       0.156391

===== Ridge alpha=1000 학습 시작 =====
alpha=1000
mean corr: 0.040060
std corr:  0.013301
sharpe:    3.011937
RMSE:      0.223104
MAE:       0.156380

===== Ridge alpha=3000 학습 시작 =====
alpha=3000
mean corr: 0.040062
std corr:  0.013292
sharpe:    3.013889
RMSE:      0.223100
MAE:       0.156346

===== Ridge alpha=10000 학습 시작 =====
alpha=10000
mean corr: 0.040067
std corr:  0.013257
sharpe:    3.022220
RMSE:      0.223088
MAE

,model,alpha,mean corr,std corr,sharpe,rmse,mae
0,Ridge,3000000,0.038823,0.011668,3.327315,0.223027,0.150554
1,Ridge,3000000,0.038823,0.011668,3.327315,0.223027,0.150554
2,Ridge,100000000,0.038150,0.011474,3.324787,0.223105,0.149712
3,Ridge,10000000,0.038162,0.011478,3.324735,0.223079,0.149976
4,Ridge,300000000,0.038150,0.011476,3.324248,0.223108,0.149693
5,Ridge,1000000000,0.038145,0.011475,3.324058,0.223108,0.149688
6,Ridge,1000000,0.039344,0.011881,3.311613,0.222955,0.151717
7,Ridge,300000,0.040030,0.012418,3.223654,0.222927,0.153596
8,Ridge,100000,0.040225,0.012712,3.164236,0.222988,0.155078
9,Ridge,30000,0.040081,0.013182,3.040570,0.223057,0.155931


In [64]:
# ElasticNet

elastic_model = SGDRegressor(
    loss="squared_error",
    penalty="elasticnet",
    alpha=1e-3,
    l1_ratio=0.03,
    learning_rate="invscaling",
    eta0=0.001,
    power_t=0.5,
    max_iter=1,
    tol=None,
    random_state=SEED,
    average=True
)

EPOCHS_ELASTIC = 20

for epoch in range(EPOCHS_ELASTIC):
    indices = np.arange(X_train_scaled.shape[0])
    np.random.shuffle(indices)

    for start in range(0, len(indices), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(indices))
        idx = indices[start:end]

        X_batch = np.array(X_train_scaled[idx], dtype=np.float32)
        y_batch = y_train_arr[idx]

        elastic_model.partial_fit(X_batch, y_batch)

    print(f"ElasticNet epoch {epoch + 1}/{EPOCHS_ELASTIC} 완료")

elastic_pred = elastic_model.predict(X_valid_scaled).astype(np.float32)

results.append(
    evaluate_and_save("elasticnet", elastic_pred)
)

gc.collect()

ElasticNet epoch 1/20 완료
ElasticNet epoch 2/20 완료
ElasticNet epoch 3/20 완료
ElasticNet epoch 4/20 완료
ElasticNet epoch 5/20 완료
ElasticNet epoch 6/20 완료
ElasticNet epoch 7/20 완료
ElasticNet epoch 8/20 완료
ElasticNet epoch 9/20 완료
ElasticNet epoch 10/20 완료
ElasticNet epoch 11/20 완료
ElasticNet epoch 12/20 완료
ElasticNet epoch 13/20 완료
ElasticNet epoch 14/20 완료
ElasticNet epoch 15/20 완료
ElasticNet epoch 16/20 완료
ElasticNet epoch 17/20 완료
ElasticNet epoch 18/20 완료
ElasticNet epoch 19/20 완료
ElasticNet epoch 20/20 완료

========== elasticnet Validation Result ==========
RMSE: 0.224640
MAE: 0.162739
per-era CORR mean: 0.040230
per-era CORR std:  0.012859
Sharpe:            3.128463
저장 완료: elasticnet_val_prediction.csv


8324

In [ ]:
# MLP
n_features = X_train_scaled.shape[1]
print("n_features:", n_features)

mlp_model = MLPRegressorTorch(input_dim=n_features).to(device)

class NumeraiDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = np.array(y).reshape(-1).astype(np.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = np.array(self.X[idx], dtype=np.float32)
        y = np.array([self.y[idx]], dtype=np.float32)
        return torch.from_numpy(x), torch.from_numpy(y)


class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("MLP device:", device)

train_dataset = NumeraiDataset(X_train_scaled, y_train)
valid_dataset = NumeraiDataset(X_valid_scaled, y_valid)

train_loader = DataLoader(
    train_dataset,
    batch_size=8192,
    shuffle=True,
    num_workers=0,
    drop_last=False
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=8192,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

mlp_model = MLPRegressorTorch(input_dim=n_features).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(
    mlp_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.8
)

EPOCHS_MLP = 15
best_valid_loss = np.inf
best_state = None
patience = 4
patience_count = 0

for epoch in range(EPOCHS_MLP):
    mlp_model.train()
    train_losses = []

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = mlp_model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    mlp_model.eval()
    valid_losses = []

    with torch.no_grad():
        for xb, yb in valid_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            pred = mlp_model(xb)
            loss = criterion(pred, yb)

            valid_losses.append(loss.item())

    train_loss = np.mean(train_losses)
    valid_loss = np.mean(valid_losses)

    scheduler.step()

    print(
        f"MLP epoch {epoch + 1}/{EPOCHS_MLP} | "
        f"train_loss: {train_loss:.6f} | valid_loss: {valid_loss:.6f}"
    )

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = mlp_model.state_dict()
        patience_count = 0
    else:
        patience_count += 1

    if patience_count >= patience:
        print("MLP early stopping")
        break

if best_state is not None:
    mlp_model.load_state_dict(best_state)

mlp_model.eval()
mlp_preds = []

with torch.no_grad():
    for xb, yb in valid_loader:
        xb = xb.to(device)
        pred = mlp_model(xb).detach().cpu().numpy().reshape(-1)
        mlp_preds.append(pred)

mlp_pred = np.concatenate(mlp_preds).astype(np.float32)

results.append(
    evaluate_and_save("mlp", mlp_pred)
)

gc.collect()


n_features: 1500
MLP device: cpu
MLP epoch 1/15 | train_loss: 0.131196 | valid_loss: 0.049995
MLP epoch 2/15 | train_loss: 0.063839 | valid_loss: 0.050924
MLP epoch 3/15 | train_loss: 0.060227 | valid_loss: 0.050341
MLP epoch 4/15 | train_loss: 0.058408 | valid_loss: 0.050000
MLP epoch 5/15 | train_loss: 0.056830 | valid_loss: 0.050077
MLP early stopping

========== mlp Validation Result ==========
RMSE: 0.223851
MAE: 0.158514
per-era CORR mean: 0.025335
per-era CORR std:  0.021187
Sharpe:            1.195758
저장 완료: mlp_val_prediction.csv


0

In [69]:
ridge_pred = ridge_model.predict(X_valid_scaled).astype(np.float32)

results.append(
    evaluate_and_save("ridge", ridge_pred)
)

elastic_pred = elastic_model.predict(X_valid_scaled).astype(np.float32)

results.append(
    evaluate_and_save("elasticnet", elastic_pred)
)
mlp_pred = np.concatenate(mlp_preds).astype(np.float32)

results.append(
    evaluate_and_save("mlp", mlp_pred)
)


========== ridge Validation Result ==========
RMSE: 0.223108
MAE: 0.149688
per-era CORR mean: 0.038145
per-era CORR std:  0.011475
Sharpe:            3.324058
저장 완료: ridge_val_prediction.csv

========== elasticnet Validation Result ==========
RMSE: 0.224640
MAE: 0.162739
per-era CORR mean: 0.040230
per-era CORR std:  0.012859
Sharpe:            3.128463
저장 완료: elasticnet_val_prediction.csv

========== mlp Validation Result ==========
RMSE: 0.223851
MAE: 0.158514
per-era CORR mean: 0.025335
per-era CORR std:  0.021187
Sharpe:            1.195758
저장 완료: mlp_val_prediction.csv
